# BoardScan — GPU training on Colab

Trains the 13-class piece CNN. Runtime → **T4 GPU**. Takes ~5 minutes.
Swap `<USER>` for your GitHub username if you forked the repo.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
!git clone https://github.com/abhishek-k-33/boardscan.git
%cd boardscan
!pip install --quiet opencv-python python-chess huggingface_hub

In [ ]:
# 1. synthetic bootstrap squares (stylised silhouettes)
!python scripts/render_synthetic_squares.py --per-class 300
# 2. realistic rendered boards (FEN in filename) -> data/raw pairs
!python -c "from huggingface_hub import hf_hub_download; hf_hub_download('honi05/chess-positions-cv', 'sample.zip', repo_type='dataset', local_dir='/content/hfdl')"
!python scripts/import_hf_sample.py --zip /content/hfdl/sample.zip --count 75
# 3. detect + slice + auto-label the realistic boards
!python scripts/build_squares_dataset.py
# 4. merge both sources
!python scripts/merge_manifests.py data/synth_squares/manifest.csv data/squares_manifest.csv -o data/combined_manifest.csv

In [ ]:
!python -u src/model/train.py --manifest data/combined_manifest.csv --epochs 25 --device auto

In [ ]:
from IPython.display import Image
Image('docs/confusion_matrix.png', width=700)

In [ ]:
from google.colab import files
files.download('models/piece_cnn.pt')  # put this in your repo's models/ folder